# JN3 · From permits to buildings — and how many homes are actually *new*

**Curriculum notebook 3 of 6.** We have a clean table of permits (JN1) and a way to tell when two addresses mean the same place (JN2). Now comes the question the whole project exists to answer: *how many new homes did Berkeley actually build?*

It sounds like a counting problem. It is really a **meaning** problem — because the single most dangerous trap in civic data hides right here: **the same column means different things on different permit types.** The `NumberUnits` field on a brand-new building is *new homes*; the very same field on a permit to "remove an electric fireplace" is the count of homes that *already existed*. Add them up without noticing, and you invent thousands of homes that were always there — or, reading too cautiously, you silently erase real ones.

By the end of this notebook you'll have turned permits into a **spine** of buildings, each with a *corrected* count of genuinely new units, and you'll have watched the trap spring in both directions on real Berkeley permits.

> Clonable + **read-only** — it demonstrates the real `housing_predicates.net_units`, never writes.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN2 · The address key](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN2_address_key.ipynb)  |  Next: [JN4 · Dated milestones + completion stage](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN4_events_stage.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Config

In [ ]:
# === CONFIG - point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


## Where do we start? Pick up exactly where JN1 and JN2 left off

Before we can count *new* homes, we need the same starting point as the earlier notebooks: the full permit feed, narrowed to the rows that are actually about **housing**. Rebuilding that by hand would risk drifting from what the pipeline really does — so instead we *import the same functions* and reproduce the exact feed.

**Our plan:** load both raw exports (JN1's move), tag each permit as New-or-not, run the real `is_housing` predicate to keep only residential rows, and confirm how many of the raw permits survive as housing.

In [ ]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units   # the REAL shared predicates (JN3 demonstrates them)
from s0_keys import normalize_address                   # the address key from JN2

# read each yearly export at the right header row, then stack them into one feed (JN1's move)
def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()              # a real permit has a number
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'   # tag the brand-new-building permits
# keep only rows the shared predicate calls housing (drops pure commercial / non-residential)
df['resi'] = [is_housing(o, u, n, a) for o, u, n, a in zip(df['OccType'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]
resi = df[df['resi']].copy()
print(f'{len(df):,} permits -> {len(resi):,} housing rows')

In [ ]:
_total = len(df); _housing = len(resi); _dropped = _total - _housing
md(f'''## What just happened

We stacked the city's raw exports into **{_total:,}** permit rows, then ran the pipeline's own `is_housing` predicate. **{_housing:,}** rows survive as housing; **{_dropped:,}** non-residential rows fall away. This is the *same* feed JN1 built and JN2 keyed — we didn't re-derive it from a teaching toy, we imported the real functions and reproduced it exactly. Everything that follows works from these **{_housing:,}** housing rows.''')

In [ ]:
import matplotlib.pyplot as plt
# how the raw feed splits into housing vs non-housing rows (reads df / resi computed above)
_housing = len(resi)
_other   = len(df) - _housing
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(['housing rows\n(is_housing = True)', 'everything else'],
        [_housing, _other], color=['#2e7d32', '#cfd8dc'])
for i, v in enumerate([_housing, _other]):
    ax.text(v, i, f' {v:,}', va='center')
ax.set_title('The housing filter: which permit rows survive into JN3')
ax.set_xlabel('permit rows'); plt.tight_layout(); plt.show()

## The corrected unit signal - `net_units` (import it, don't reinvent)

`housing_predicates.net_units(is_new, units_added, number_units, adu_flag)` is the **one** rule the whole pipeline uses. Read its four branches - each exists because of a real bug it prevents:

| branch | rule | why |
|---|---|---|
| 1 | `UnitsAdded` if `>0` | the explicit *net add* - trust it when present |
| 2 | else `NumberUnits` if **New** | a new building: *all* its units are new |
| 3 | else `min(NumberUnits, 2)` if **ADU** | the ADU/JADU - but **capped at 2**, because a permit on an 82-unit building flagged ADU reports `NumberUnits=82` (the existing stock), not 82 new ADUs |
| 4 | **else `0`** | a plain alteration's `NumberUnits` is the **existing** count - **not new housing** |

In [ ]:
print(net_units.__doc__)

## THE CORE LESSON: same column, different meaning

So what goes wrong if you just trust `NumberUnits`? Here is the trap in one sentence. On a **New** permit, `NumberUnits` is *new units*. On an **Alteration**, it is the count of the **building that already exists**. The obvious rule — "use `UnitsAdded`, else fall back to `NumberUnits`" — cannot tell those two cases apart, so it counts existing stock as if every home were brand new.

The cleanest way to *feel* the bug is to watch it on a single real permit: **B2023-02847**, an alteration at **2024 Durant** whose entire scope of work was *"remove electric fireplace, replace light fixtures"* — in a building that **already had 99 units**. The naive rule will happily report 99 new homes for a fireplace removal.

**Our plan:** run both rules on this one permit and watch the naive rule invent phantom homes while `net_units` (branch 4) correctly returns zero.

In [ ]:
r = df[df['PermitNumber'] == 'B2023-02847'].iloc[0]
ua, nu = float(r['UnitsAdded'] or 0), float(r['NumberUnits'] or 0)
naive   = ua if ua > 0 else nu                                  # 'UnitsAdded, else NumberUnits'
correct = net_units(r['isnew'], r['UnitsAdded'], r['NumberUnits'], r['ADU'])
print(f"B2023-02847  WorkType={r['Work Type']}  NumberUnits={r['NumberUnits']}  UnitsAdded={r['UnitsAdded']}")
print(f"  work: {str(r['WorkDescription'])[:70]}")
print(f'  NAIVE  ua-else-nu     -> {naive:.0f} new units   <-- 99 PHANTOM homes (the building already existed)')
print(f'  net_units (branch 4)  -> {correct:.0f} new units   <-- correct: an alteration adds no new housing')

In [ ]:
_wt = r['Work Type']; _nu = r['NumberUnits']; _ua = r['UnitsAdded']
md(f'''## What just happened

One permit, two rules, two very different answers. B2023-02847 is an **{_wt}** with `NumberUnits={_nu}` and `UnitsAdded={_ua}`. The naive "UnitsAdded-else-NumberUnits" rule has no `UnitsAdded` to lean on, so it falls back to `NumberUnits` and reports **{naive:.0f}** new homes — for a permit whose only work was removing a fireplace. Those **{naive:.0f}** homes already existed; they are pure phantoms.

`net_units` reads the *work type* first. It sees an alteration (not New, not an ADU), lands on **branch 4**, and returns **{correct:.0f}**. An alteration creates no new housing, so the corrected count is zero. Same column, opposite meaning — and the rule that survives is the one that asks *what kind of permit is this?* before trusting the number.''')

In [ ]:
import matplotlib.pyplot as plt
# the fireplace permit, two rules side by side (reads naive / correct computed above)
fig, ax = plt.subplots(figsize=(5.5, 3))
bars = ax.bar(['naive\n(ua-else-nu)', 'net_units\n(branch 4)'],
              [naive, correct], color=['#c0392b', '#2e7d32'])
ax.bar_label(bars, fmt='%.0f', padding=3)
ax.set_title('B2023-02847 (remove a fireplace): "new" homes by rule')
ax.set_ylabel('new units the rule reports')
ax.set_ylim(0, max(naive, 1) * 1.25); plt.tight_layout(); plt.show()

### How big is the trap, really? Apply both rules to the whole feed

One fireplace permit invented 99 phantom homes. But is that a rare freak, or a systematic leak? The only way to know is to stop reasoning about one row and *measure* it across every housing permit.

**Our plan:** sum the naive rule and `net_units` over the entire housing feed, and read off the gap — the total number of homes the naive rule conjures out of existing stock. (The pipeline later groups permits to buildings with **MAX**, not SUM — a lesson still ahead — which absorbs most of this, but phantom units still leak into the spine if the per-permit rule is wrong.)

In [ ]:
def fnum(x):
    try: v = float(str(x).replace(',', ''))
    except: return 0.0
    return v if v == v else 0.0          # NaN guard (a blank cell reads as 'nan')
def nz(v): return v if v == v else 0.0
# sum each rule across every housing permit: the naive fallback vs the corrected net_units
naive_permit   = sum(fnum(r.UnitsAdded) if fnum(r.UnitsAdded) > 0 else fnum(r.NumberUnits) for r in resi.itertuples())
correct_permit = sum(nz(net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU)) for r in resi.itertuples())
print(f'per-permit, summed across the feed:  naive {naive_permit:,.0f}  vs  net_units {correct_permit:,.0f}')
print(f'  -> {naive_permit - correct_permit:,.0f} phantom units the naive rule invents from existing stock')

In [ ]:
_phantom = naive_permit - correct_permit
_ratio = naive_permit / correct_permit if correct_permit else float('nan')
md(f'''## What just happened

The fireplace was not a freak. Summed across every housing permit, the naive rule reports **{naive_permit:,.0f}** units; `net_units` reports **{correct_permit:,.0f}**. The difference — **{_phantom:,.0f}** phantom homes — is existing stock the naive rule mistakes for new construction. That is the naive count running about **{_ratio:.1f}×** too high.

This is why a per-permit rule that "looks reasonable" is so dangerous: it runs without error and produces a number that is wrong by tens of thousands. The fix isn't a smarter spreadsheet formula — it's reading the *meaning* of the column per permit type, which is exactly what `net_units` does.''')

In [ ]:
import matplotlib.pyplot as plt
# feed-wide totals: the phantom gap is the red slice the naive rule adds on top of the real count
_phantom = naive_permit - correct_permit
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(['naive\n(ua-else-nu)'], [correct_permit], color='#2e7d32', label='real net-new units')
ax.bar(['naive\n(ua-else-nu)'], [_phantom], bottom=[correct_permit], color='#c0392b', label='phantom (existing stock)')
ax.bar(['net_units\n(corrected)'], [correct_permit], color='#2e7d32')
ax.set_title('Phantom homes: the naive rule vs net_units, summed over the whole feed')
ax.set_ylabel('units reported across all housing permits'); ax.legend()
plt.tight_layout(); plt.show()

## The trap runs the other way too: too-cautious a rule silently DROPS real housing

We just saw over-counting. But could you over-correct? Suppose, burned by the fireplace, you swing the other way and trust *only* `UnitsAdded`. That feels conservative and safe — until you realize ADUs (backyard cottages, in-law units) are coded with their count in **`NumberUnits`**, with `UnitsAdded=0`. A `UnitsAdded`-only rule reads every one of them as **zero** and quietly erases real homes that were genuinely built.

`net_units` catches these on **branch 3** (`min(NumberUnits, 2)` when the ADU flag is set). A perfect example: **B2021-03756** at 1534 Oregon — `ADU=Yes`, `NumberUnits=2`, `UnitsAdded=0`.

**Our plan:** run the cautious `UnitsAdded`-only rule on this ADU, watch it drop the home, then watch branch 3 recover it — and count how many such ADUs live in the feed.

In [ ]:
r = df[df['PermitNumber'] == 'B2021-03756'].iloc[0]
ua_only = float(r['UnitsAdded'] or 0)                                # the over-cautious rule: UnitsAdded only
correct = net_units(r['isnew'], r['UnitsAdded'], r['NumberUnits'], r['ADU'])
print(f"B2021-03756  ADU={r['ADU']}  NumberUnits={r['NumberUnits']}  UnitsAdded={r['UnitsAdded']}")
print(f'  UnitsAdded-only -> {ua_only:.0f}  <-- a real ADU DROPPED')
print(f'  net_units (branch 3, min(nu,2)) -> {correct:.0f}  <-- recovered')
# count the ADU tail: ADU-flagged, count in NumberUnits, no UnitsAdded, not a New build
n_adu = sum(1 for r in resi.itertuples() if str(r.ADU).strip().lower() == 'yes'
            and fnum(r.NumberUnits) > 0 and fnum(r.UnitsAdded) == 0 and not r.isnew)
print(f'\nADU permits whose count lives in NumberUnits (recovered by branch 3): {n_adu}  (the ~265-ADU tail)')

In [ ]:
_adu = r['ADU']; _nu = r['NumberUnits']
md(f'''## What just happened

The over-cautious rule fails in the opposite direction. B2021-03756 is a real ADU (`ADU={_adu}`, `NumberUnits={_nu}`), but its count lives in `NumberUnits`, not `UnitsAdded`. A `UnitsAdded`-only rule reads it as **{ua_only:.0f}** and erases the home; branch 3 recovers it as **{correct:.0f}**.

And it is not one stray cottage — **{n_adu}** permits in this feed are ADUs whose count would vanish under a `UnitsAdded`-only rule. `net_units` has to be exactly as wide as reality: narrow enough to reject existing stock (branch 4), wide enough to keep real ADUs (branch 3). Both traps, one rule.''')

## Now build the spine: how do permits become *buildings*?

A permit is not a building. A single building can carry many permits — the original New permit, revisions, phased filings. To count *buildings*, we have to group permits that belong together, and to count their homes we have to pick one number per group without double-counting.

What groups them? The **address key** from JN2. What number do we keep? The **MAX** `net_units` across the group — never the SUM, because phased permits (`Phase I`, `Phase II`) each repeat the *whole-building* count, so summing would multiply it. A building earns a place in the **spine** if it has any New permit or any net-new units.

**Our plan:** key every housing permit by its address, fold permits onto buildings keeping the MAX corrected unit count, drop buildings with no new housing, and confirm the result reproduces the pipeline's S1 spine.

In [ ]:
spine = defaultdict(lambda: {'units': 0.0, 'hasnew': False})
for r in resi.itertuples(index=False):
    # build the address key (drop a missing/blank street type before keying)
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    key = (k.number, k.street, k.stype)
    # fold this permit onto its building: keep the MAX corrected unit count (never SUM)
    spine[key]['units'] = max(spine[key]['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: spine[key]['hasnew'] = True
# a building is in the spine only if it has a New permit or any net-new units
spine = {k: b for k, b in spine.items() if b['hasnew'] or b['units'] > 0}
print(f"spine buildings: {len(spine)}   (matches the pipeline's S1 spine of 1385)")
print(f"spine net-new units: {sum(b['units'] for b in spine.values()):,.0f}")

In [ ]:
_n = len(spine); _u = sum(b['units'] for b in spine.values())
_big = sum(1 for b in spine.values() if b['units'] >= 50)
_small = sum(1 for b in spine.values() if 0 < b['units'] <= 2)
md(f'''## What just happened

Folding {len(resi):,} housing permits onto their address keys yields **{_n}** buildings — exactly the pipeline's S1 spine of 1385, so our reconstruction matches production. Together they hold **{_u:,.0f}** net-new units, each one a *corrected* count: phantom existing-stock kept out, real ADUs kept in.

The histogram tells the shape of Berkeley's housing: **{_small}** buildings add just 1–2 units (the ADU/small-infill tail), while a handful of large developments — **{_big}** buildings of 50+ units — carry most of the homes. Using MAX (not SUM) is what kept those large buildings from being multiplied by their own phased permits.''')

In [ ]:
import matplotlib.pyplot as plt
# distribution of building sizes in the spine (reads the spine dict computed above)
_sizes = [b['units'] for b in spine.values()]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(_sizes, bins=range(0, 60, 2), color='#1565c0', edgecolor='white')
ax.set_title('Spine building sizes: most are small, a long tail of large buildings')
ax.set_xlabel('net-new units in the building (clipped at 60 for the view)')
ax.set_ylabel('number of buildings')
plt.tight_layout(); plt.show()

## A known limitation, on purpose: what does keying by address miss?

The spine is good — but no derivation is perfect, and the honest move is to name where ours bends. Here is the question to ask of *any* grouping rule: what happens when the thing you grouped on isn't unique? We keyed buildings by **address**. So what about two genuinely different buildings that share one address?

That is exactly **2352 Shattuck** — the Logan Park development, which is a 135-unit North building *and* a separate 69-unit South building on the same address. Our address key collapses them into one, and MAX keeps only the larger.

**Our plan:** look at 2352 Shattuck in the spine, see that only the larger building survives, and name this as a *taught* limitation you will rediscover and fix in JN6 — not a bug we're hiding.

In [ ]:
k = normalize_address('2352 Shattuck Ave')
print(f'2352 Shattuck in this spine: {spine[(k.number, k.street, k.stype)]}')
print('  -> ONE building, 135u (the North). The South 69u is hidden by MAX-on-one-address.')
print('  -> This is NOT fixed here. You will REDISCOVER it in JN6 when the scorecard residual')
print('     surfaces it -- a collapse made into a taught lesson, not a hidden bug.')

In [ ]:
_logan = spine[(k.number, k.street, k.stype)]
md(f'''## What just happened

2352 Shattuck shows up in the spine as **one** building of **{_logan['units']:.0f}** units — the North tower. The 69-unit South building is real, finished, and *gone* from this count, hidden because MAX over a single address key keeps only the larger of two co-located buildings.

We are deliberately **not** patching this here. A fix bolted on now would be invisible magic; instead the collapse stays in the open, and JN6's scorecard residual will surface it as a discrepancy you trace back to this exact cause. That is the difference between a hidden bug and a taught limitation — we wrote it down.''')

## The checkpoint: verify before you trust

We've corrected the unit signal and built the spine — but how do we *know* it stayed right? Four cheap assertions pin the whole notebook's contract down: the spine reproduces the pipeline's building count, the fireplace alteration still adds zero new homes (the over-counting guard), the ADU still contributes its units (the under-counting guard), and the ADU tail survives in bulk. If someone later "simplifies" `net_units` or the grouping, one of these breaks loudly — *now*, not three notebooks downstream.

In [ ]:
# 1) the spine reproduces the pipeline's S1 building count
assert len(spine) == 1385, f'spine {len(spine)} != 1385'
# 2) the else-0 guard: a real alteration adds 0 new units (existing stock NOT counted)
alt = df[df['PermitNumber'] == 'B2023-02847'].iloc[0]
assert net_units(alt['isnew'], alt['UnitsAdded'], alt['NumberUnits'], alt['ADU']) == 0
# 3) the ADU tail is present (not dropped): a NumberUnits-coded ADU contributes its units
adu = df[df['PermitNumber'] == 'B2021-03756'].iloc[0]
assert net_units(adu['isnew'], adu['UnitsAdded'], adu['NumberUnits'], adu['ADU']) == 2
assert n_adu > 250   # the ADU tail survives

print('CHECKPOINT PASS')
print(f'  spine = {len(spine)} buildings (== S1)  -  alteration B2023-02847 -> 0 new (else-0 guard)')
print(f'  ADU tail present: {n_adu} NumberUnits-coded ADUs recovered (1534 Oregon -> 2 units)')

In [ ]:
_alt_nu = net_units(alt['isnew'], alt['UnitsAdded'], alt['NumberUnits'], alt['ADU'])
_adu_nu = net_units(adu['isnew'], adu['UnitsAdded'], adu['NumberUnits'], adu['ADU'])
md(f'''## What just happened

All four assertions passed. The spine holds **{len(spine)}** buildings (== the pipeline's S1). The fireplace alteration B2023-02847 still scores **{_alt_nu:.0f}** new units — the over-counting guard holds. The ADU B2021-03756 still scores **{_adu_nu:.0f}** — the under-counting guard holds. And **{n_adu}** NumberUnits-coded ADUs survive in the feed.

That is the whole habit of this course in one cell: **state what must be true, then make the computer prove it.** This spine — buildings with a corrected unit count — is the foundation JN4 attaches dated milestones to.''')

**JN3 done.** You have a spine of buildings with a *corrected* unit count - phantom existing-stock kept out, real ADUs kept in. **Next - JN4:** dated milestone events and the completion stage (when is a building actually *done*?).

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN2 · The address key](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN2_address_key.ipynb)  |  Next: [JN4 · Dated milestones + completion stage](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN4_events_stage.ipynb) →